# Capítulo 6 · Algoritmo de Shor

## Objetivos

1. Entender la reducción del problema de factorización al de encontrar el orden de $a$ módulo $N$.
2. Implementar el circuito de estimación de fase para la factorización de $N = 15$.
3. Realizar el post-procesamiento clásico (fracciones continuas) para extraer el orden.
4. Verificar que se obtienen los factores correctos.

---

## 6.1 Esquema del algoritmo

El algoritmo de Shor factoriza $N$ en tiempo $O((\log N)^3)$ combinando:

**Paso clásico previo:** Elegir $a$ aleatorio con $1 < a < N$ y $\gcd(a, N) = 1$.  
**Paso cuántico:** Encontrar el orden $r$ de $a$ módulo $N$, es decir, el menor $r > 0$ con $a^r \equiv 1 \pmod{N}$.  
**Paso clásico posterior:** Si $r$ es par y $a^{r/2} \not\equiv -1 \pmod{N}$, entonces

$$p = \gcd(a^{r/2} - 1, N) \quad \text{y} \quad q = \gcd(a^{r/2} + 1, N)$$

son factores no triviales de $N$.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from math import gcd
from fractions import Fraction
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Operator, Statevector
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Dependencias cargadas.')

## 6.2 Post-procesamiento clásico

In [ ]:
def classical_postprocessing(measured_phase: int, n_count: int,
                              N: int, a: int) -> dict:
    """Post-procesamiento del algoritmo de Shor.

    Aplica el algoritmo de fracciones continuas a la fase medida para
    obtener el orden r, y luego calcula los factores de N.

    Parámetros
    ----------
    measured_phase : int
        Entero medido en los qubits de conteo.
    n_count : int
        Número de qubits de conteo (precisión = 2^n_count).
    N : int
        Número a factorizar.
    a : int
        Base elegida.

    Retorna
    -------
    dict con claves: phase_ratio, r, factors, success.
    """
    # Fase estimada
    phase_ratio = measured_phase / (2 ** n_count)

    # Aproximación por fracción continua
    frac = Fraction(phase_ratio).limit_denominator(N)
    r = frac.denominator

    result = {
        'phase_ratio': phase_ratio,
        'r': r,
        'factors': [],
        'success': False,
    }

    if r == 0 or r % 2 != 0:
        return result

    # Candidatos a factores
    ar2 = pow(a, r // 2, N)
    if ar2 == N - 1:   # a^{r/2} ≡ -1 (mod N)
        return result

    p = gcd(ar2 - 1, N)
    q = gcd(ar2 + 1, N)

    factors = [f for f in [p, q] if 1 < f < N]
    result['factors'] = factors
    result['success'] = len(factors) > 0

    return result


# Ejemplo manual: a=2, N=15, r=4 (conocido)
res = classical_postprocessing(measured_phase=4, n_count=3, N=15, a=2)
print('Post-procesamiento (ejemplo manual):')
for k, v in res.items():
    print(f'  {k}: {v}')

## 6.3 Circuito cuántico de estimación de fase para N=15, a=2

Para $N=15$ y $a=2$, el orden es $r=4$ porque $2^4 = 16 \equiv 1 \pmod{15}$. Implementamos el oráculo de multiplicación modular $|x\rangle \mapsto |ax \bmod N\rangle$ para este caso concreto.

In [ ]:
def c_amod15(a: int, power: int) -> QuantumCircuit:
    """Puerta controlada U^{2^power} para a mod 15.

    Sólo implementa los casos a ∈ {2, 4, 7, 8, 11, 13}.
    """
    if a not in [2, 4, 7, 8, 11, 13]:
        raise ValueError(f'a={a} no implementado.')

    U = QuantumCircuit(4, name=f'U_{a}^{2**power}')
    # Aplicar power veces la permutación correspondiente
    for _ in range(power):
        if a == 2:
            U.swap(0, 1); U.swap(1, 2); U.swap(2, 3)
        elif a == 7:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        elif a == 8:
            U.swap(2, 3); U.swap(1, 2); U.swap(0, 1)
        elif a == 4:
            U.swap(1, 3); U.swap(0, 2)
        elif a == 11:
            U.swap(0, 2); U.swap(1, 3)
            U.x(range(4))
        elif a == 13:
            U.swap(0, 3); U.swap(1, 2)
            U.x(range(4))

    U_gate = U.to_gate().control(1)
    qc = QuantumCircuit(5)
    qc.append(U_gate, range(5))
    return qc


def shor_circuit_n15(a: int = 2, n_count: int = 8) -> QuantumCircuit:
    """Circuito de Shor para N=15.

    Parámetros
    ----------
    a : int
        Base (debe ser coprima con 15).
    n_count : int
        Número de qubits de conteo (más qubits = más precisión).
    """
    from notebooks.ch04_fourier_cuantica import qft  # importar localmente
    # importar la función QFT definida aquí
    pass


# ── Implementación directa con Qiskit ──────────────────────────────
def build_shor_n15(a: int = 2, n_count: int = 8) -> QuantumCircuit:
    """Circuito de Shor para N=15 (implementación directa)."""
    from qiskit.circuit.library import QFT

    # Registro de conteo (n_count qubits) + registro de trabajo (4 qubits)
    q_count = QuantumRegister(n_count, 'count')
    q_aux   = QuantumRegister(4, 'aux')
    cr      = ClassicalRegister(n_count, 'meas')
    qc = QuantumCircuit(q_count, q_aux, cr)

    # Estado inicial del registro auxiliar = |1〉
    qc.x(q_aux[0])

    # Hadamard en qubits de conteo
    qc.h(q_count)
    qc.barrier()

    # Puertas U^{2^j} controladas
    for j in range(n_count):
        controlled_U = c_amod15(a, 2**j)
        # conectar qubit de conteo j con el registro auxiliar
        qc.append(controlled_U, [q_count[j]] + list(q_aux))
    qc.barrier()

    # QFT inversa sobre el registro de conteo
    iqft = QFT(n_count, inverse=True, do_swaps=True)
    qc.append(iqft, q_count)

    # Medida
    qc.measure(q_count, cr)
    return qc


n_count = 8
a = 2
qc_shor = build_shor_n15(a=a, n_count=n_count)
print(f'Circuito de Shor para N=15, a={a}, n_count={n_count}:')
print(f'  Profundidad: {qc_shor.depth()}')
print(f'  Número de operaciones: {qc_shor.size()}')

In [ ]:
# Ejecutar el circuito
backend = AerSimulator()
job = backend.run(qc_shor, shots=2048)
counts = job.result().get_counts()

print(f'Resultados (top 10):')
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])[:10]
for state, cnt in sorted_counts:
    phase_int = int(state, 2)
    result = classical_postprocessing(phase_int, n_count, N=15, a=a)
    factors_str = f' → factores: {result["factors"]}' if result['success'] else ''
    print(f'  {state} (={phase_int:3d}): {cnt:4d} ocurrencias{factors_str}')

fig = QuantumVisualization.plot_histogram(
    counts,
    title=f'Shor N=15, a={a}: distribución de fases medidas',
    color='#d2a8ff',
)
plt.show()

## 6.4 Ejercicios propuestos

1. Repite el experimento con $a = 7$ y $a = 13$ para $N = 15$. ¿Obtienes los mismos factores?

2. Explica por qué el algoritmo clásico de fracciones continuas puede fallar y cuántas repeticiones cuánticas son necesarias en promedio para garantizar el éxito.

3. ¿Cuántos qubits físicos se necesitarían para factorizar RSA-2048 con el algoritmo de Shor, asumiendo corrección de errores cuánticos?

4. Implementa el paso de reducción clásica completo: dado $N$ arbitrario, elige $a$ aleatorio y verifica si $\gcd(a, N) \neq 1$ antes de llamar al circuito cuántico.